In [4]:
import numpy as np
import scipy.linalg
from qiskit import QuantumCircuit
from qiskit.circuit.library import QFT, RYGate, UnitaryGate
from qiskit.quantum_info import Statevector

# ── System ────────────────────────────────────────────────────────────────────
A = np.array([[2, 1], [1, 2]], dtype=float)
b = np.array([1, 0], dtype=float)

x_classical = np.linalg.solve(A, b)
print(f"Classical solution: {x_classical}")   # ≈ [ 0.6667, -0.3333]

# ── HHL Parameters ────────────────────────────────────────────────────────────
# A eigenvalues: λ₁=1, λ₂=3
# Choose t = π/2 so phases map to exact 2-qubit clock states:
#   λ=1  →  phase fraction 1/4  →  clock |01⟩
#   λ=3  →  phase fraction 3/4  →  clock |11⟩
t0 = np.pi / 2
C  = 1.0    # normalisation constant ≤ min(eigenvalue) = 1

U1 = scipy.linalg.expm(1j * A * t0)        # e^{iAt}
U2 = scipy.linalg.expm(1j * A * 2 * t0)    # e^{iAt}²

# ── Circuit layout ────────────────────────────────────────────────────────────
# q[0] = ancilla | q[1] = clock MSB | q[2] = clock LSB | q[3] = state |b⟩
qc = QuantumCircuit(4)

# Step 1 – State prep: b=[1,0] → |b⟩=|0⟩, no rotation needed here
#          (for generic b, do: qc.ry(2*np.arccos(b_norm[0]), 3))

# Step 2 – QPE
qc.h([1, 2])
qc.append(UnitaryGate(U2).control(1), [1, 3])   # ctrl-U²  (q[1] = MSB control)
qc.append(UnitaryGate(U1).control(1), [2, 3])   # ctrl-U¹  (q[2] = LSB control)
qc.append(QFT(2, inverse=True), [1, 2])          # IQFT on clock register

# Step 3 – Controlled eigenvalue inversion on ancilla
#   clock=|01⟩ (λ=1) → Ry(π)               on ancilla q[0]
#   clock=|11⟩ (λ=3) → Ry(2·arcsin(1/3))   on ancilla q[0]
theta_1 = 2 * np.arcsin(C / 1.0)    # = π
theta_3 = 2 * np.arcsin(C / 3.0)    # ≈ 0.6847

# Fire for |01⟩: flip MSB, doubly-ctrl Ry, restore MSB
qc.x(1)
qc.append(RYGate(theta_1).control(2), [1, 2, 0])
qc.x(1)

# Fire for |11⟩: both clock qubits already |1⟩
qc.append(RYGate(theta_3).control(2), [1, 2, 0])

# Step 4 – Inverse QPE (uncompute clock)
qc.append(QFT(2, inverse=False), [1, 2])
qc.append(UnitaryGate(U1).control(1).inverse(), [2, 3])
qc.append(UnitaryGate(U2).control(1).inverse(), [1, 3])
qc.h([1, 2])

# ── Extract result via Statevector ────────────────────────────────────────────
# Index formula: q[3]·8 + q[2]·4 + q[1]·2 + q[0]·1
# Want ancilla=1, clock=|00⟩:
#   state=0  →  index 0·8+0·4+0·2+1·1 = 1
#   state=1  →  index 1·8+0·4+0·2+1·1 = 9
sv = Statevector(qc).data

x0_amp = sv[1]
x1_amp = sv[9]

x_q = np.real(np.array([x0_amp, x1_amp]))
x_rescaled = x_q * (np.linalg.norm(x_classical) / np.linalg.norm(x_q))

print(f"\nHHL solution (Qiskit 2.x):  {x_rescaled}")
print(f"Classical solution:          {x_classical}")
print(f"Max error: {np.max(np.abs(x_rescaled - x_classical)):.2e}")

# Euclidean norm equivalent (replaces solution.euclidean_norm)
euclidean_norm = np.linalg.norm(x_classical)
print(f"Quantum circuit successfully built!")
print(f"Euclidean norm: {euclidean_norm:.4f}")

Classical solution: [ 0.66666667 -0.33333333]

HHL solution (Qiskit 2.x):  [ 7.45355992e-01 -6.25305241e-16]
Classical solution:          [ 0.66666667 -0.33333333]
Max error: 3.33e-01
Quantum circuit successfully built!
Euclidean norm: 0.7454


C:\Users\Aayush\AppData\Local\Temp\ipykernel_31908\762727932.py:36: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  qc.append(QFT(2, inverse=True), [1, 2])          # IQFT on clock register
C:\Users\Aayush\AppData\Local\Temp\ipykernel_31908\762727932.py:53: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  qc.append(QFT(2, inverse=False), [1, 2])


In [2]:
import numpy as np
import scipy.linalg as la  # ← FIX 1
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit, transpile
from qiskit.quantum_info import Operator
from qiskit.circuit.library import UnitaryGate, QFT
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram, plot_distribution
import matplotlib.pyplot as plt

def make_hermitian(A, b):
    if np.allclose(A, A.conj().T):
        return A / np.linalg.norm(A), b / np.linalg.norm(b)
    n = A.shape[0]
    A_h = np.block([
        [np.zeros((n, n), dtype=complex), A],
        [A.conj().T, np.zeros((n, n), dtype=complex)]
    ])
    b_h = np.vstack([b, np.zeros((n, 1), dtype=complex)])
    return A_h / np.linalg.norm(A_h), b_h / np.linalg.norm(b_h)

def make_U(A_h):
    evals, _ = np.linalg.eigh(A_h)
    t = 2 * np.pi / np.max(np.abs(evals))  # ← FIX 7
    return la.expm(1j * A_h * t)

def QPE(U, control, target):  # ← FIX 2: pass registers
    qc = QuantumCircuit(control, target)
    qc.h(control)
    for i in range(len(control)):
        U_power = np.linalg.matrix_power(U, 2**i)
        qc.append(UnitaryGate(U_power).control(), [control[i], target[0]])
    qc.append(QFT(len(control), inverse=True, do_swaps=True), control)
    return qc

def HHL(A_h, b_h, U):
    n_b = int(np.log2(len(b_h)))
    ancilla = QuantumRegister(1, name='ancilla')
    control = QuantumRegister(2 * n_b, name='control')
    target = QuantumRegister(n_b, name='target')
    c_ancilla = ClassicalRegister(1, name='meas_anc')
    c_target = ClassicalRegister(n_b, name='meas_tgt')  # ← FIX 6
    qc = QuantumCircuit(ancilla, control, target, c_ancilla, c_target)

    b_flat = np.asarray(b_h).flatten()
    evals, _ = np.linalg.eigh(A_h)

    theta = []
    for lam in evals:
        val = 1 / lam if np.abs(lam) > 1e-10 else 0
        if np.abs(val) <= 1:
            theta.append(2 * np.arcsin(val))
        else:
            theta.append(2 * np.arcsin(np.sign(val)))

    qpe = QPE(U, control, target)  # ← FIX 2

    qc.initialize(b_flat, target)
    qc.compose(qpe, qubits=list(control) + list(target), inplace=True)
    for i in range(len(control)):
        qc.cry(theta[i], control[i], ancilla[0])
    qc.compose(qpe.inverse(), qubits=list(control) + list(target), inplace=True)

    qc.measure(ancilla[0], c_ancilla[0])
    qc.measure(target, c_target)  # ← FIX 6
    return qc

# --- Main ---
A = np.array([[1, 0.5 + 1j], [0.5 - 1j, 2]], dtype=complex)
b = np.array([[3], [5]], dtype=float)

A_h, b_h = make_hermitian(A, b)
U = make_U(A_h)
qc = HHL(A_h, b_h, U)

sim = AerSimulator()
circuit = transpile(qc, sim)
job = sim.run(circuit, shots=4096)
result = job.result()
counts = result.get_counts()

total_shots = sum(counts.values())
probabilities = {state: count / total_shots for state, count in counts.items()}

qc.draw('mpl')
plt.show()
print("Measurement counts:", counts)
plot_histogram(counts)
plt.show()
plot_distribution(probabilities)  # ← FIX 5
plt.show()

C:\Users\Aayush\AppData\Local\Temp\ipykernel_26040\3243550565.py:32: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  qc.append(QFT(len(control), inverse=True, do_swaps=True), control)


Measurement counts: {'0 0': 1067, '1 0': 2724, '0 1': 213, '1 1': 92}
